In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from vocal_assistant.vocal_assistant import VocalAssistant
from vocal_assistant.emotion.emotion_model import process_func, EmotionModel
from transformers import Wav2Vec2Processor
from vocal_assistant.emotion.predict_emotion import load_trained_model, predict_emotion
import pandas as pd


In [ ]:
songs = pd.read_pickle("./data/Songs")


In [2]:
vc = VocalAssistant(1)

In [3]:
device = 'cpu'

audeering_model_name = 'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
audeering_processor = Wav2Vec2Processor.from_pretrained(audeering_model_name)
audeering_model = EmotionModel.from_pretrained(audeering_model_name).to(device)

custom_model_name = "model_checkpoint_sampled.pth"
custom_model, custom_processor = load_trained_model(custom_model_name)

/Users/lucabellani/Documents/UNI/Tesi/Recommersion/vocal_assistant/emotion/predict_emotion.py:198: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(chec

Loaded trained model from checkpoint.


In [7]:
vc.talk("What is your mood today?")
while True:
    command, vocal_file = vc.take_command()
    print(command)
    break

print("Audeering: ")
audeering = process_func(vocal_file, 16000)[0]
print(audeering)
print("Custom: ")
#custom = list(predict_emotion(custom_model, custom_processor, vocal_file).values())
custom = predict_emotion(custom_model, custom_processor, vocal_file)[0].tolist()
print(custom)
#custom model seems to give the same results: overfitting?
# [0.5256028771400452, 0.6051671504974365, 0.5843760371208191]


  listening....
Sample rate: 16000
Numpy array shape: (38267,)
i am sad
Audeering: 
[-0.15744051  0.03438079  0.0665469 ]
Custom: 
[0.35754257440567017, 0.746627926826477, 0.6955891251564026]


: 

In [5]:
import numpy as np
dim_vec = np.array(audeering[0:2])
songs_list = pd.DataFrame({"id": songs["musicId"], "eucl_dist":songs[["Valence", "Arousal"]]\
                           .apply(lambda x: np.linalg.norm(x - dim_vec), axis=1), "Valence": songs["Valence"], "Arousal": songs["Arousal"],\
                            "title":songs["title"], "artist": songs["artist"], "mp3_file":songs["mp3_file"]})

songs_list = songs_list.sort_values(by="eucl_dist")[:5]
songs_list

NameError: name 'songs' is not defined

In [ ]:
import sounddevice as sd

for i in range(len(songs_list)):
    sd.play(songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()


KeyboardInterrupt: 